In [1]:
import sys
from pathlib import Path
ROOT = Path.cwd() if (Path.cwd() / "src").exists() else Path.cwd().parent
sys.path.append(str(ROOT))

import pandas as pd
from sklearn.linear_model import LogisticRegression

from src.data import load_raw, add_target, get_X_y
from src.preprocessing import build_pipeline, get_feature_types
from src.config import RANDOM_STATE, N_SPLITS

df = add_target(load_raw("por"))
X, y = get_X_y(df, "a_no_grades")   # start with the hardest, most useful scenario
print("X:", X.shape, "| y:", y.shape)

X: (649, 30) | y: (649,)


In [2]:
pipe = build_pipeline(LogisticRegression(max_iter=1000, class_weight="balanced"), X)

# Fit ONLY the preprocessing step, just to inspect its output
X_ready = pipe.named_steps["prep"].fit_transform(X)
feature_names = pipe.named_steps["prep"].get_feature_names_out()

num_cols, cat_cols = get_feature_types(X)
print(f"Raw features: {X.shape[1]}  ({len(num_cols)} numeric + {len(cat_cols)} categorical)")
print(f"After preprocessing: {X_ready.shape[1]} columns\n")
print("Example new column names:", list(feature_names[13:22]))

# Check the scaling worked: numeric columns should have mean ~0, std ~1
pd.DataFrame(X_ready[:, :len(num_cols)], columns=num_cols).describe().loc[["mean", "std"]].round(2)

Raw features: 30  (13 numeric + 17 categorical)
After preprocessing: 43 columns

Example new column names: ['school_MS', 'sex_M', 'address_U', 'famsize_LE3', 'Pstatus_T', 'Mjob_at_home', 'Mjob_health', 'Mjob_other', 'Mjob_services']


,age,Medu,Fedu,traveltime,studytime,failures,famrel,freetime,goout,Dalc,Walc,health,absences
mean,-0.0,-0.0,-0.0,-0.0,0.0,0.0,-0.0,0.0,0.0,-0.0,0.0,-0.0,0.0
std,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0


In [3]:
from sklearn.model_selection import StratifiedKFold, cross_validate

# Stratified = every fold keeps the same ~15% fail ratio as the full data
cv = StratifiedKFold(n_splits=N_SPLITS, shuffle=True, random_state=RANDOM_STATE)

scores = cross_validate(pipe, X, y, cv=cv, scoring=["recall", "f1", "roc_auc"])

for metric in ["recall", "f1", "roc_auc"]:
    s = scores[f"test_{metric}"]
    print(f"{metric:8}: {s.mean():.3f}  (± {s.std():.3f} across {N_SPLITS} folds)")

recall  : 0.670  (± 0.150 across 5 folds)
f1      : 0.465  (± 0.074 across 5 folds)
roc_auc : 0.793  (± 0.060 across 5 folds)
